In [3]:
import pandas as pd
import numpy as np
import seaborn as sns
import torch 

Tensors

1. Used in the input and output of models
2. Like numpy nd arrays but can use GPUS


In [ ]:
#Directly from a nested array
data1 = [[1,2,3],[4,5,6],[7,8,9]]
torch_data_1 = torch.tensor(data1)

#From a numpy array
np_array = np.array(data1)
torch_data_2 = torch.from_numpy(np_array)

#Keeps the same structure from torch_data_1
torch_data1_ones = torch.ones_like(torch_data_1)
print(torch_data1_ones)

#Overides the datatype
data_rand = torch.rand_like(torch_data_1, dtype=torch.float) # overrides the datatype of x_data
print(f"Random Tensor: \n {data_rand} \n")


tensor([[1, 1, 1],
        [1, 1, 1],
        [1, 1, 1]])
Random Tensor: 
 tensor([[0.9796, 0.2636, 0.4511],
        [0.8839, 0.3287, 0.5158],
        [0.4904, 0.3589, 0.9148]]) 



In [ ]:
tensor = torch.ones(4, 4)
tensor[:,1] = 0
print(tensor)

#tensor concatenation
t1 = torch.cat([tensor, tensor, tensor], dim=1)
print(t1)


tensor([[1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.]])
tensor([[1., 0., 1., 1., 1., 0., 1., 1., 1., 0., 1., 1.],
        [1., 0., 1., 1., 1., 0., 1., 1., 1., 0., 1., 1.],
        [1., 0., 1., 1., 1., 0., 1., 1., 1., 0., 1., 1.],
        [1., 0., 1., 1., 1., 0., 1., 1., 1., 0., 1., 1.]])


In [9]:
# This computes the element-wise product (index * index)
print(f"tensor.mul(tensor) \n {tensor.mul(tensor)} \n")
# Alternative syntax:
print(f"tensor * tensor \n {tensor * tensor}")

tensor.mul(tensor) 
 tensor([[1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.]]) 

tensor * tensor 
 tensor([[1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.]])


In [ ]:
#tensor matrix multiplication
print(f"tensor.matmul(tensor.T) \n {tensor.matmul(tensor.T)} \n")
# Alternative syntax:
print(f"tensor @ tensor.T \n {tensor @ tensor.T}")

In [ ]:
#In place operations include "_" changes the tensor directly without making a copy.
print(tensor, "\n")
tensor.add_(5)
print(tensor)
#Could save memory, but can be problematic when computing gradients in neural networks.

In [ ]:
#numpy back from tensor:
t = tensor.new_ones(5)
print(f"t: {t}")
t.numpy()
print(f"n: {n}")

In [ ]:
#Changes in tensor are also applied onto the numpy arrays and vice versa.
t.add_(1)
print(f"t: {t}")
print(f"n: {n}")

2. torch autograd
https://docs.pytorch.org/tutorials/beginner/blitz/autograd_tutorial.html 

In [ ]:
from torchvision.models import resnet18, ResNet18_Weights
model = resnet18(weights=ResNet18_Weights.DEFAULT)
data = torch.rand(1, 3, 64, 64)
labels = torch.rand(1, 1000)

prediction = model(data) # forward pass

loss = (prediction - labels).sum()

loss.backward() # backward pass - Stores gradient in the parameters of the model

optim = torch.optim.SGD(model.parameters(), lr=1e-2, momentum=0.9)

optim.step() #gradient descent FOR ONE STEP

In [ ]:
import torchvision
from torchvision.transform import v2

1. Loading and Normalizing the data set v2

In [ ]:
transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

batch_size = 4

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size,
                                          shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size,
                                         shuffle=False, num_workers=2)

classes = ('plane', 'car', 'bird', 'cat',
           'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# functions to show an image


def imshow(img):
    img = img / 2 + 0.5     # unnormalize
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.show()


# get some random training images
dataiter = iter(trainloader)
images, labels = next(dataiter)

# show images
imshow(torchvision.utils.make_grid(images))
# print labels
print(' '.join(f'{classes[labels[j]]:5s}' for j in range(batch_size)))

2. Defining the NN

In [ ]:
import torch.nn as nn
import torch.nn.functional as F


class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 6, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1) # flatten all dimensions except batch
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x


net = Net()

3. Defining the Loss and optimizer

In [ ]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)

4. Trainning the network

In [ ]:
for epoch in range(2):  # loop over the dataset multiple times

    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        # get the inputs; data is a list of [inputs, labels]
        inputs, labels = data

        # zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # print statistics
        running_loss += loss.item()
        if i % 2000 == 1999:    # print every 2000 mini-batches
            print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 2000:.3f}')
            running_loss = 0.0

print('Finished Training')

5. Saving the model

In [ ]:
PATH = './cifar_net.pt'
torch.save(net.state_dict(), PATH)